In [201]:
import pandas as pd
import numpy as np
import re
import json


In [202]:
emissions_data_batched = pd.read_csv('emissions_batched_benchmakrs.csv')
emissions_data_nonbatched = pd.read_csv('emissions_benchmakrs.csv')

leaderboard_path = 'arena-hard-auto/leaderboard/arena_hard_leaderboard_20240819.csv'
leaderboard_df = pd.read_csv(leaderboard_path)

In [203]:
# Clean the leaderboard DataFrame
# Extract CI values from the CI column and add them as new columns
leaderboard_df['CI_lower'] = leaderboard_df['CI'].apply(lambda x: float(x.split(',')[0].strip('("(').strip()))
leaderboard_df['CI_upper'] = leaderboard_df['CI'].apply(lambda x: float(x.split(',')[1].strip(')+"').strip()))

# Rename columns to avoid conflicts
leaderboard_df = leaderboard_df.rename(columns={
    'model': 'model_name',
    'avg_tokens': 'output_tok',
    'score': 'arena_score',
    'rating_q025': '95_conf_minus',
    'rating_q975': '95_conf_plus'
})


In [204]:
leaderboard_df

,model_name,arena_score,95_conf_minus,95_conf_plus,CI,output_tok,date,CI_lower,CI_upper
0,gpt-4o,88.09,86.46,89.50,"(-1.63, +1.41)",696.0,2024-08-19,-1.63,1.41
1,gpt-4o-2024-08-06_guided,87.75,86.11,89.28,"(-1.64, +1.53)",625.0,2024-08-19,-1.64,1.53
2,gpt-4o-2024-08-06,86.55,85.15,88.04,"(-1.40, +1.49)",616.0,2024-08-19,-1.40,1.49
3,gpt-4o-mini_guided,86.27,84.69,87.83,"(-1.58, +1.56)",678.0,2024-08-19,-1.58,1.56
4,mistral_large_2_guided,83.63,81.88,85.48,"(-1.75, +1.85)",777.0,2024-08-19,-1.75,1.85
5,mistral_large_2,80.96,79.36,82.80,"(-1.60, +1.84)",684.0,2024-08-19,-1.60,1.84
6,gpt-4o-mini,80.09,78.41,81.95,"(-1.68, +1.86)",631.0,2024-08-19,-1.68,1.86
7,llama3_1_405b_guided,79.47,77.59,81.40,"(-1.88, +1.93)",649.0,2024-08-19,-1.88,1.93
8,llama3_1_70b_guided,78.29,76.37,80.16,"(-1.92, +1.87)",646.0,2024-08-19,-1.92,1.87
9,llama3_1_70b_fp8_dyn_guided,75.14,73.12,77.16,"(-2.02, +2.02)",646.0,2024-08-19,-2.02,2.02


In [205]:
input_tok_map = {
    'llama3_1': 135.142,
    'llama3': 135.142,
    'mistral_nemo': 143.19,
}

input_tok_map_guided = {
    'llama3_1': 874.592,
    'llama3': 874.592,
    'mistral_nemo': 921.216,
}

In [206]:
emissions_data_batched.head(5)

,timestamp,project_name,run_id,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,...,cpu_count,cpu_model,gpu_count,gpu_model,longitude,latitude,ram_total_size,tracking_mode,on_cloud,pue
0,2024-08-14T19:43:28,llama3_1_8b_fp8-arena-hard-v0.1-4gpus-run_1,024763a9-9f4d-42ec-9cf1-0442d6329e2d,365.810330,0.027216,0.000074,42.5,272.583341,68.162159,0.005269,...,48,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765759,machine,N,1.22
1,2024-08-14T19:49:15,llama3_1_8b_fp8-arena-hard-v0.1-4gpus-run_2,9856a69f-e1b6-41f7-b334-206aee430ee1,331.955029,0.025369,0.000076,42.5,279.438916,68.162159,0.004781,...,48,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765759,machine,N,1.22
2,2024-08-14T19:54:57,llama3_1_8b_fp8-arena-hard-v0.1-4gpus-run_3,34c856fd-c8be-45c8-ae34-4c46f4376f86,326.117697,0.025017,0.000077,42.5,275.269343,68.162159,0.004697,...,48,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765759,machine,N,1.22
3,2024-08-15T14:20:14,mistral_nemo-arena-hard-v0.1-4gpus-run_1,7d4bba3c-3d08-47f3-ae02-4cec3ee2fb99,454.586133,0.037299,0.000082,42.5,288.397371,68.162164,0.006547,...,48,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765770,machine,N,1.22
4,2024-08-15T14:28:17,mistral_nemo-arena-hard-v0.1-4gpus-run_2,757fd0d2-462e-45e8-b59c-22e0cd69772f,468.639597,0.039750,0.000085,42.5,288.776447,68.162164,0.006750,...,48,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765770,machine,N,1.22


In [207]:
# Strip "-run_x" and "-xgpus" and "-arena-hard-v0.1" from the project_name and create a new column for model_name
emissions_data_batched['model_name'] = emissions_data_batched['project_name'].apply(lambda x: re.sub(r'-run_\d+', '', re.sub(r'-arena-hard-v0.1', '', x)))

In [208]:
emissions_data_batched.head(5)

,timestamp,project_name,run_id,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,...,cpu_model,gpu_count,gpu_model,longitude,latitude,ram_total_size,tracking_mode,on_cloud,pue,model_name
0,2024-08-14T19:43:28,llama3_1_8b_fp8-arena-hard-v0.1-4gpus-run_1,024763a9-9f4d-42ec-9cf1-0442d6329e2d,365.810330,0.027216,0.000074,42.5,272.583341,68.162159,0.005269,...,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765759,machine,N,1.22,llama3_1_8b_fp8-4gpus
1,2024-08-14T19:49:15,llama3_1_8b_fp8-arena-hard-v0.1-4gpus-run_2,9856a69f-e1b6-41f7-b334-206aee430ee1,331.955029,0.025369,0.000076,42.5,279.438916,68.162159,0.004781,...,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765759,machine,N,1.22,llama3_1_8b_fp8-4gpus
2,2024-08-14T19:54:57,llama3_1_8b_fp8-arena-hard-v0.1-4gpus-run_3,34c856fd-c8be-45c8-ae34-4c46f4376f86,326.117697,0.025017,0.000077,42.5,275.269343,68.162159,0.004697,...,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765759,machine,N,1.22,llama3_1_8b_fp8-4gpus
3,2024-08-15T14:20:14,mistral_nemo-arena-hard-v0.1-4gpus-run_1,7d4bba3c-3d08-47f3-ae02-4cec3ee2fb99,454.586133,0.037299,0.000082,42.5,288.397371,68.162164,0.006547,...,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765770,machine,N,1.22,mistral_nemo-4gpus
4,2024-08-15T14:28:17,mistral_nemo-arena-hard-v0.1-4gpus-run_2,757fd0d2-462e-45e8-b59c-22e0cd69772f,468.639597,0.039750,0.000085,42.5,288.776447,68.162164,0.006750,...,AMD EPYC 7R13 Processor,4,4 x NVIDIA L4,-83.0061,39.9625,181.765770,machine,N,1.22,mistral_nemo-4gpus


In [209]:
emissions_data_nonbatched['model_name'] = emissions_data_nonbatched['project_name'].apply(lambda x: re.sub(r'-arena-hard-v0.1', '', x))

In [210]:
emissions_data_nonbatched.head(5)

,timestamp,project_name,run_id,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,...,cpu_model,gpu_count,gpu_model,longitude,latitude,ram_total_size,tracking_mode,on_cloud,pue,model_name
0,2024-08-10T22:02:33,llama3_1_70b-arena-hard-v0.1-8gpus,3aadccc3-c126-4029-b145-64d8026d62f3,2099.365285,0.389977,0.000186,42.5,554.894717,273.079609,0.030237,...,AMD EPYC 7R13 Processor,8,8 x NVIDIA L4,-83.0061,39.9625,728.212292,machine,N,1.22,llama3_1_70b-8gpus
1,2024-08-11T16:28:27,llama3_1_70b-arena-hard-v0.1-8gpus,4e7850b1-9134-4db3-b7a9-57881a6268c2,2138.534305,0.401009,0.000188,42.5,536.236980,273.079634,0.030801,...,AMD EPYC 7R13 Processor,8,8 x NVIDIA L4,-83.0061,39.9625,728.212357,machine,N,1.22,llama3_1_70b-8gpus
2,2024-08-11T18:35:04,llama3_70b_guided-arena-hard-v0.1-8gpus,14719f66-dafe-41a5-96ee-8d6a4b9b2230,2963.909088,0.554097,0.000187,42.5,578.761208,273.079634,0.042688,...,AMD EPYC 7R13 Processor,8,8 x NVIDIA L4,-83.0061,39.9625,728.212357,machine,N,1.22,llama3_70b_guided-8gpus


In [211]:
# Group by the new model_name and calculate the mean for the selected columns
df_batched = emissions_data_batched.groupby('model_name').agg({
    'duration': 'mean',
    'emissions': 'mean',
    'emissions_rate': 'mean',
    'cpu_power': 'mean',
    'gpu_power': 'mean',
    'ram_power': 'mean',
    'cpu_energy': 'mean',
    'gpu_energy': 'mean',
    'ram_energy': 'mean',
    'energy_consumed': 'mean',
    'gpu_count': 'first'}).reset_index()
    
df_nonbatched = emissions_data_nonbatched.groupby('model_name').agg({
    'duration': 'mean',
    'emissions': 'mean',
    'emissions_rate': 'mean',
    'cpu_power': 'mean',
    'gpu_power': 'mean',
    'ram_power': 'mean',
    'cpu_energy': 'mean',
    'gpu_energy': 'mean',
    'ram_energy': 'mean',
    'energy_consumed': 'mean',
    'gpu_count': 'first'}).reset_index()

In [212]:
df_batched.head(5)

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,energy_consumed,gpu_count
0,llama3_1_70b-8gpus,2396.648948,0.435146,0.000182,42.5,576.966292,273.079595,0.034518,0.398597,0.221290,0.654406,8
1,llama3_1_70b_awq_int4-4gpus,1480.617988,0.132385,0.000089,42.5,288.748807,68.162162,0.021325,0.143604,0.034161,0.199090,4
2,llama3_1_70b_awq_int4_guided-4gpus,2009.356246,0.177109,0.000088,42.5,288.028506,68.162162,0.028940,0.191046,0.046363,0.266350,4
3,llama3_1_70b_fp8-4gpus,2002.761728,0.179008,0.000089,42.5,288.561002,68.162162,0.028845,0.194161,0.046200,0.269207,4
4,llama3_1_70b_fp8-8gpus,1220.240308,0.211623,0.000173,42.5,581.537797,273.079614,0.017575,0.187966,0.112714,0.318255,8


In [213]:
df_nonbatched.head(5)

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,energy_consumed,gpu_count
0,llama3_1_70b-8gpus,2118.949795,0.395493,0.000187,42.5,545.565849,273.079622,0.030519,0.368565,0.195689,0.594773,8
1,llama3_70b_guided-8gpus,2963.909088,0.554097,0.000187,42.5,578.761208,273.079634,0.042688,0.516876,0.273729,0.833293,8


In [214]:
df = pd.concat([df_batched, df_nonbatched], ignore_index=True)
df = df.groupby('model_name').agg({
    'duration': 'mean',
    'emissions': 'mean',
    'emissions_rate': 'mean',
    'cpu_power': 'mean',
    'gpu_power': 'mean',
    'ram_power': 'mean',
    'cpu_energy': 'mean',
    'gpu_energy': 'mean',
    'ram_energy': 'mean',
    'energy_consumed': 'mean',
    'gpu_count': 'first'}).reset_index()
df.head(10)

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,energy_consumed,gpu_count
0,llama3_1_70b-8gpus,2257.799372,0.415319,0.000184,42.5,561.266070,273.079608,0.032519,0.383581,0.208489,0.624589,8
1,llama3_1_70b_awq_int4-4gpus,1480.617988,0.132385,0.000089,42.5,288.748807,68.162162,0.021325,0.143604,0.034161,0.199090,4
2,llama3_1_70b_awq_int4_guided-4gpus,2009.356246,0.177109,0.000088,42.5,288.028506,68.162162,0.028940,0.191046,0.046363,0.266350,4
3,llama3_1_70b_fp8-4gpus,2002.761728,0.179008,0.000089,42.5,288.561002,68.162162,0.028845,0.194161,0.046200,0.269207,4
4,llama3_1_70b_fp8-8gpus,1220.240308,0.211623,0.000173,42.5,581.537797,273.079614,0.017575,0.187966,0.112714,0.318255,8
5,llama3_1_70b_fp8_guided-8gpus,1882.436985,0.321359,0.000171,42.5,579.537203,273.079614,0.027112,0.282304,0.173868,0.483285,8
6,llama3_1_70b_guided-8gpus,3240.161237,0.571359,0.000176,42.5,1403.876058,273.079595,0.046667,0.513398,0.299188,0.859254,8
7,llama3_1_70b_int4-8gpus,980.887494,0.154287,0.000157,42.5,529.552211,273.079595,0.014127,0.127292,0.090609,0.232029,8
8,llama3_1_70b_int4_guided-8gpus,1730.090809,0.273492,0.000158,42.5,539.607969,273.079595,0.024918,0.226562,0.159818,0.411298,8
9,llama3_1_8b-4gpus,502.168702,0.041314,0.000082,42.5,287.894549,68.162161,0.007233,0.043311,0.011587,0.062131,4


In [215]:
df['model_id'] = df['model_name']
df['model_name'] = df['model_name'].apply(lambda x: re.sub(r'-\d+gpus', '', x))
df['model_name'] = df['model_name'].apply(lambda x: re.sub(r'_awq', '', x))
leaderboard_df['model_name'] = leaderboard_df['model_name'].apply(lambda x: re.sub(r'_dyn', '', x))
leaderboard_df['model_name'] = leaderboard_df['model_name'].apply(lambda x: re.sub(r'_awq', '', x))

In [216]:
df

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,energy_consumed,gpu_count,model_id
0,llama3_1_70b,2257.799372,0.415319,0.000184,42.5,561.266070,273.079608,0.032519,0.383581,0.208489,0.624589,8,llama3_1_70b-8gpus
1,llama3_1_70b_int4,1480.617988,0.132385,0.000089,42.5,288.748807,68.162162,0.021325,0.143604,0.034161,0.199090,4,llama3_1_70b_awq_int4-4gpus
2,llama3_1_70b_int4_guided,2009.356246,0.177109,0.000088,42.5,288.028506,68.162162,0.028940,0.191046,0.046363,0.266350,4,llama3_1_70b_awq_int4_guided-4gpus
3,llama3_1_70b_fp8,2002.761728,0.179008,0.000089,42.5,288.561002,68.162162,0.028845,0.194161,0.046200,0.269207,4,llama3_1_70b_fp8-4gpus
4,llama3_1_70b_fp8,1220.240308,0.211623,0.000173,42.5,581.537797,273.079614,0.017575,0.187966,0.112714,0.318255,8,llama3_1_70b_fp8-8gpus
5,llama3_1_70b_fp8_guided,1882.436985,0.321359,0.000171,42.5,579.537203,273.079614,0.027112,0.282304,0.173868,0.483285,8,llama3_1_70b_fp8_guided-8gpus
6,llama3_1_70b_guided,3240.161237,0.571359,0.000176,42.5,1403.876058,273.079595,0.046667,0.513398,0.299188,0.859254,8,llama3_1_70b_guided-8gpus
7,llama3_1_70b_int4,980.887494,0.154287,0.000157,42.5,529.552211,273.079595,0.014127,0.127292,0.090609,0.232029,8,llama3_1_70b_int4-8gpus
8,llama3_1_70b_int4_guided,1730.090809,0.273492,0.000158,42.5,539.607969,273.079595,0.024918,0.226562,0.159818,0.411298,8,llama3_1_70b_int4_guided-8gpus
9,llama3_1_8b,502.168702,0.041314,0.000082,42.5,287.894549,68.162161,0.007233,0.043311,0.011587,0.062131,4,llama3_1_8b-4gpus


In [217]:
# Perform an outer join to include all models from both DataFrames
df = pd.merge(df, leaderboard_df, on='model_name', how='outer')

# Extract models that have no energy data (all columns from the original df will be NaN)
models_with_no_energy_data = df[df['duration'].isna()]['model_name'].unique()

# Extract models that have no arena score data (all columns from the leaderboard_df will be NaN)
models_with_no_arena_score = df[df['arena_score'].isna()]['model_name'].unique()

df['model_id'] = df['model_id'].fillna(df['model_name'])

df.head(10)

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,...,gpu_count,model_id,arena_score,95_conf_minus,95_conf_plus,CI,output_tok,date,CI_lower,CI_upper
0,gpt-4-0314,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,gpt-4-0314,50.00,50.00,50.00,"(-0.00, +0.00)",423.0,2024-08-19,-0.00,0.00
1,gpt-4o,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,gpt-4o,88.09,86.46,89.50,"(-1.63, +1.41)",696.0,2024-08-19,-1.63,1.41
2,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,gpt-4o-2024-08-06,86.55,85.15,88.04,"(-1.40, +1.49)",616.0,2024-08-19,-1.40,1.49
3,gpt-4o-2024-08-06_guided,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,gpt-4o-2024-08-06_guided,87.75,86.11,89.28,"(-1.64, +1.53)",625.0,2024-08-19,-1.64,1.53
4,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,gpt-4o-mini,80.09,78.41,81.95,"(-1.68, +1.86)",631.0,2024-08-19,-1.68,1.86
5,gpt-4o-mini_guided,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,gpt-4o-mini_guided,86.27,84.69,87.83,"(-1.58, +1.56)",678.0,2024-08-19,-1.58,1.56
6,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,llama3_1_405b,72.20,69.85,74.40,"(-2.35, +2.20)",662.0,2024-08-19,-2.35,2.20
7,llama3_1_405b_guided,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,llama3_1_405b_guided,79.47,77.59,81.40,"(-1.88, +1.93)",649.0,2024-08-19,-1.88,1.93
8,llama3_1_70b,2257.799372,0.415319,0.000184,42.5,561.266070,273.079608,0.032519,0.383581,0.208489,...,8.0,llama3_1_70b-8gpus,68.05,65.90,70.28,"(-2.15, +2.23)",681.0,2024-08-19,-2.15,2.23
9,llama3_1_70b_fp8,2002.761728,0.179008,0.000089,42.5,288.561002,68.162162,0.028845,0.194161,0.046200,...,4.0,llama3_1_70b_fp8-4gpus,68.79,66.79,71.25,"(-2.00, +2.46)",689.0,2024-08-19,-2.00,2.46


In [218]:
no_energy = pd.DataFrame({"no energy data": models_with_no_energy_data})
no_energy

,no energy data
0,gpt-4-0314
1,gpt-4o
2,gpt-4o-2024-08-06
3,gpt-4o-2024-08-06_guided
4,gpt-4o-mini
5,gpt-4o-mini_guided
6,llama3_1_405b
7,llama3_1_405b_guided
8,mistral_large_2
9,mistral_large_2_guided


In [219]:
no_arena = pd.DataFrame({"no arena score": models_with_no_arena_score})
no_arena

,no arena score


In [220]:
# Create the guided column
df['guided'] = df['model_name'].apply(lambda x: 1 if '_guided' in x else 0)

# Strip "_guided" from the model name
df['model_name'] = df['model_name'].apply(lambda x: re.sub(r'_guided', '', x))

In [221]:
df.head(5)

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,...,model_id,arena_score,95_conf_minus,95_conf_plus,CI,output_tok,date,CI_lower,CI_upper,guided
0,gpt-4-0314,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,gpt-4-0314,50.00,50.00,50.00,"(-0.00, +0.00)",423.0,2024-08-19,-0.00,0.00,0
1,gpt-4o,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,gpt-4o,88.09,86.46,89.50,"(-1.63, +1.41)",696.0,2024-08-19,-1.63,1.41,0
2,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,gpt-4o-2024-08-06,86.55,85.15,88.04,"(-1.40, +1.49)",616.0,2024-08-19,-1.40,1.49,0
3,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,gpt-4o-2024-08-06_guided,87.75,86.11,89.28,"(-1.64, +1.53)",625.0,2024-08-19,-1.64,1.53,1
4,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,gpt-4o-mini,80.09,78.41,81.95,"(-1.68, +1.86)",631.0,2024-08-19,-1.68,1.86,0


In [222]:
# Add a new column 'quantization' based on the suffixes in the model name
def assign_quantization(model_name):
    if '_fp8' in model_name:
        return 'fp8'
    elif '_fp8_dyn' in model_name:
        return 'fp8'
    elif '_awq_int4' in model_name:
        return '_int4'
    elif '_int4' in model_name:
        return 'int4'
    else:
        return 'bf16'

In [223]:
df['quantization'] = df['model_name'].apply(assign_quantization)

# Strip the quantization suffixes from the model name
df['model_name'] = df['model_name'].apply(lambda x: re.sub(r'_fp8_dyn|_fp8|_awq_int4|_int4', '', x))

In [224]:
df.head(10)

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,...,arena_score,95_conf_minus,95_conf_plus,CI,output_tok,date,CI_lower,CI_upper,guided,quantization
0,gpt-4-0314,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,50.00,50.00,50.00,"(-0.00, +0.00)",423.0,2024-08-19,-0.00,0.00,0,bf16
1,gpt-4o,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,88.09,86.46,89.50,"(-1.63, +1.41)",696.0,2024-08-19,-1.63,1.41,0,bf16
2,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,86.55,85.15,88.04,"(-1.40, +1.49)",616.0,2024-08-19,-1.40,1.49,0,bf16
3,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,87.75,86.11,89.28,"(-1.64, +1.53)",625.0,2024-08-19,-1.64,1.53,1,bf16
4,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,80.09,78.41,81.95,"(-1.68, +1.86)",631.0,2024-08-19,-1.68,1.86,0,bf16
5,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,86.27,84.69,87.83,"(-1.58, +1.56)",678.0,2024-08-19,-1.58,1.56,1,bf16
6,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,72.20,69.85,74.40,"(-2.35, +2.20)",662.0,2024-08-19,-2.35,2.20,0,bf16
7,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,79.47,77.59,81.40,"(-1.88, +1.93)",649.0,2024-08-19,-1.88,1.93,1,bf16
8,llama3_1_70b,2257.799372,0.415319,0.000184,42.5,561.266070,273.079608,0.032519,0.383581,0.208489,...,68.05,65.90,70.28,"(-2.15, +2.23)",681.0,2024-08-19,-2.15,2.23,0,bf16
9,llama3_1_70b,2002.761728,0.179008,0.000089,42.5,288.561002,68.162162,0.028845,0.194161,0.046200,...,68.79,66.79,71.25,"(-2.00, +2.46)",689.0,2024-08-19,-2.00,2.46,0,fp8


In [225]:
# Function to extract model class and remove the suffix from the model name
def extract_param_size(model_name):
    match = re.search(r'_\d+b', model_name)
    if match:
        param_size = match.group(0).lstrip('_')
        param_size = param_size.rstrip('b')
        return int(param_size)
    elif model_name == 'mistral_nemo':
        return 12
    elif model_name == 'mistral_large_2':
        return 123
    return np.nan

In [226]:
df['param_size'] = df['model_name'].apply(extract_param_size)
df['model_class'] = df['model_name'].apply(lambda x: re.sub(r'_\d+b', '', x))

df

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,...,95_conf_plus,CI,output_tok,date,CI_lower,CI_upper,guided,quantization,param_size,model_class
0,gpt-4-0314,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,50.00,"(-0.00, +0.00)",423.0,2024-08-19,-0.00,0.00,0,bf16,NaN,gpt-4-0314
1,gpt-4o,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,89.50,"(-1.63, +1.41)",696.0,2024-08-19,-1.63,1.41,0,bf16,NaN,gpt-4o
2,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,88.04,"(-1.40, +1.49)",616.0,2024-08-19,-1.40,1.49,0,bf16,NaN,gpt-4o-2024-08-06
3,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,89.28,"(-1.64, +1.53)",625.0,2024-08-19,-1.64,1.53,1,bf16,NaN,gpt-4o-2024-08-06
4,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,81.95,"(-1.68, +1.86)",631.0,2024-08-19,-1.68,1.86,0,bf16,NaN,gpt-4o-mini
5,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,87.83,"(-1.58, +1.56)",678.0,2024-08-19,-1.58,1.56,1,bf16,NaN,gpt-4o-mini
6,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,74.40,"(-2.35, +2.20)",662.0,2024-08-19,-2.35,2.20,0,bf16,405.0,llama3_1
7,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,81.40,"(-1.88, +1.93)",649.0,2024-08-19,-1.88,1.93,1,bf16,405.0,llama3_1
8,llama3_1_70b,2257.799372,0.415319,0.000184,42.5,561.266070,273.079608,0.032519,0.383581,0.208489,...,70.28,"(-2.15, +2.23)",681.0,2024-08-19,-2.15,2.23,0,bf16,70.0,llama3_1
9,llama3_1_70b,2002.761728,0.179008,0.000089,42.5,288.561002,68.162162,0.028845,0.194161,0.046200,...,71.25,"(-2.00, +2.46)",689.0,2024-08-19,-2.00,2.46,0,fp8,70.0,llama3_1


In [227]:
def map_input_tok(row):
    if row['guided'] == 1:
        return input_tok_map_guided.get(row['model_class'], 0)
    else:
        return input_tok_map.get(row['model_class'], 0)

In [228]:
df['input_tok'] = df.apply(map_input_tok, axis=1)

In [229]:
df['duration_minutes'] = df['duration'] / 60
df

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,...,output_tok,date,CI_lower,CI_upper,guided,quantization,param_size,model_class,input_tok,duration_minutes
0,gpt-4-0314,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,423.0,2024-08-19,-0.00,0.00,0,bf16,NaN,gpt-4-0314,0.000,NaN
1,gpt-4o,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,696.0,2024-08-19,-1.63,1.41,0,bf16,NaN,gpt-4o,0.000,NaN
2,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,616.0,2024-08-19,-1.40,1.49,0,bf16,NaN,gpt-4o-2024-08-06,0.000,NaN
3,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,625.0,2024-08-19,-1.64,1.53,1,bf16,NaN,gpt-4o-2024-08-06,0.000,NaN
4,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,631.0,2024-08-19,-1.68,1.86,0,bf16,NaN,gpt-4o-mini,0.000,NaN
5,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,678.0,2024-08-19,-1.58,1.56,1,bf16,NaN,gpt-4o-mini,0.000,NaN
6,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,662.0,2024-08-19,-2.35,2.20,0,bf16,405.0,llama3_1,135.142,NaN
7,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,649.0,2024-08-19,-1.88,1.93,1,bf16,405.0,llama3_1,874.592,NaN
8,llama3_1_70b,2257.799372,0.415319,0.000184,42.5,561.266070,273.079608,0.032519,0.383581,0.208489,...,681.0,2024-08-19,-2.15,2.23,0,bf16,70.0,llama3_1,135.142,37.629990
9,llama3_1_70b,2002.761728,0.179008,0.000089,42.5,288.561002,68.162162,0.028845,0.194161,0.046200,...,689.0,2024-08-19,-2.00,2.46,0,fp8,70.0,llama3_1,135.142,33.379362


In [230]:
df['num_prompts'] = 500
df['sec_per_prompt'] = df['duration'] / df['num_prompts']
df

,model_name,duration,emissions,emissions_rate,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,...,CI_lower,CI_upper,guided,quantization,param_size,model_class,input_tok,duration_minutes,num_prompts,sec_per_prompt
0,gpt-4-0314,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-0.00,0.00,0,bf16,NaN,gpt-4-0314,0.000,NaN,500,NaN
1,gpt-4o,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.63,1.41,0,bf16,NaN,gpt-4o,0.000,NaN,500,NaN
2,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.40,1.49,0,bf16,NaN,gpt-4o-2024-08-06,0.000,NaN,500,NaN
3,gpt-4o-2024-08-06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.64,1.53,1,bf16,NaN,gpt-4o-2024-08-06,0.000,NaN,500,NaN
4,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.68,1.86,0,bf16,NaN,gpt-4o-mini,0.000,NaN,500,NaN
5,gpt-4o-mini,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.58,1.56,1,bf16,NaN,gpt-4o-mini,0.000,NaN,500,NaN
6,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-2.35,2.20,0,bf16,405.0,llama3_1,135.142,NaN,500,NaN
7,llama3_1_405b,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-1.88,1.93,1,bf16,405.0,llama3_1,874.592,NaN,500,NaN
8,llama3_1_70b,2257.799372,0.415319,0.000184,42.5,561.266070,273.079608,0.032519,0.383581,0.208489,...,-2.15,2.23,0,bf16,70.0,llama3_1,135.142,37.629990,500,4.515599
9,llama3_1_70b,2002.761728,0.179008,0.000089,42.5,288.561002,68.162162,0.028845,0.194161,0.046200,...,-2.00,2.46,0,fp8,70.0,llama3_1,135.142,33.379362,500,4.005523


In [231]:
df.dtypes

model_name           object
duration            float64
emissions           float64
emissions_rate      float64
cpu_power           float64
gpu_power           float64
ram_power           float64
cpu_energy          float64
gpu_energy          float64
ram_energy          float64
energy_consumed     float64
gpu_count           float64
model_id             object
arena_score         float64
95_conf_minus       float64
95_conf_plus        float64
CI                   object
output_tok          float64
date                 object
CI_lower            float64
CI_upper            float64
guided                int64
quantization         object
param_size          float64
model_class          object
input_tok           float64
duration_minutes    float64
num_prompts           int64
sec_per_prompt      float64
dtype: object

In [232]:
new_column_order = [
    'model_id',
    'model_name',
    'model_class',        
    'param_size', 
    'gpu_count',        
    'quantization',       
    'guided',             
    'num_prompts',        
    'output_tok',         
    'input_tok',          
    'arena_score',
    'CI', 
    'CI_lower',
    'CI_upper',        
    '95_conf_plus',       
    '95_conf_minus',      
    'duration',
    'duration_minutes',
    'sec_per_prompt',
    'cpu_power',
    'gpu_power',
    'ram_power',
    'cpu_energy',
    'gpu_energy',
    'ram_energy',
    'energy_consumed',     
    'emissions',
    'emissions_rate'
]

df = df[new_column_order]

df

,model_id,model_name,model_class,param_size,gpu_count,quantization,guided,num_prompts,output_tok,input_tok,...,sec_per_prompt,cpu_power,gpu_power,ram_power,cpu_energy,gpu_energy,ram_energy,energy_consumed,emissions,emissions_rate
0,gpt-4-0314,gpt-4-0314,gpt-4-0314,NaN,NaN,bf16,0,500,423.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,gpt-4o,gpt-4o,gpt-4o,NaN,NaN,bf16,0,500,696.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,gpt-4o-2024-08-06,gpt-4o-2024-08-06,gpt-4o-2024-08-06,NaN,NaN,bf16,0,500,616.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,gpt-4o-2024-08-06_guided,gpt-4o-2024-08-06,gpt-4o-2024-08-06,NaN,NaN,bf16,1,500,625.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,gpt-4o-mini,gpt-4o-mini,gpt-4o-mini,NaN,NaN,bf16,0,500,631.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,gpt-4o-mini_guided,gpt-4o-mini,gpt-4o-mini,NaN,NaN,bf16,1,500,678.0,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,llama3_1_405b,llama3_1_405b,llama3_1,405.0,NaN,bf16,0,500,662.0,135.142,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,llama3_1_405b_guided,llama3_1_405b,llama3_1,405.0,NaN,bf16,1,500,649.0,874.592,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,llama3_1_70b-8gpus,llama3_1_70b,llama3_1,70.0,8.0,bf16,0,500,681.0,135.142,...,4.515599,42.5,561.266070,273.079608,0.032519,0.383581,0.208489,0.624589,0.415319,0.000184
9,llama3_1_70b_fp8-4gpus,llama3_1_70b,llama3_1,70.0,4.0,fp8,0,500,689.0,135.142,...,4.005523,42.5,288.561002,68.162162,0.028845,0.194161,0.046200,0.269207,0.179008,0.000089


In [233]:
# Define the relative path to the target directory
save_path = '../results/data/results.parquet'

# Save the DataFrame to the Parquet file
df.to_parquet(save_path, index=False)